In [2]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [5]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv", encoding="latin-1")

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [7]:
from sklearn.model_selection import train_test_split

In [9]:
X = data["text"]
y = data["label"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## Data Preprocessing

In [11]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [13]:
import re

In [20]:
text = X_train.iloc[0]
re.sub("<.*?>", "", text)
cleaned_X_train = [re.sub("<.*?>", '', text) for text in X_train]
print(cleaned_X_train)

['When you are ready pls call into ops and they will connect u to blair.', "From=3AMr=2EKabir Bello=2E E-mail=3A Kabir=40she=2EcomTel=3A+66-1-4919460=2EAttention=3AThe Managing DirectorBefore I proceed=2C may I humbly introduce myself to your good self=2C My Name is Mr=2EKabir Bello=2C an Iraqi refugee =2Cmy Father was until recently=2C one of the personal aid to the president of Iraq who was formerly overthrown out of power during the War in My Country which took many life =2E Prior to this last serious crisis that is still ravaging in my country=2C which recently led to misfortune of our government and my late Father's position as the personal aid to the president=2C we were able to come over here in Thailand with my mother for investment =2Cwe inherited the sum of US$16 Million=2E The funds were originally gotten from my late Father's proceeds=2E My late father was able to safe guard the fund with a very good diplomatic contact from my Country to Thailand which make us to come here 

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [32]:
cleaned_X_train = [re.sub(r'[^a-zA-Z0-9\s]','',text)for text in cleaned_X_train]
cleaned_X_train = [re.sub(r'[0-9]', '', text) for text in cleaned_X_train]
cleaned_X_train = [re.sub(r'\b[a-zA-Z]\b', '', text) for text in cleaned_X_train]
cleaned_X_train = [re.sub(r'^[a-zA-Z]\s+', '',text) for text in cleaned_X_train]
cleaned_X_train = [re.sub(r'\s+', ' ', text)for text in cleaned_X_train]
cleaned_X_train = [re.sub(r'^','', text) for text in cleaned_X_train]
cleaned_X_train = [text.lower() for text in cleaned_X_train]

## Now let's work on removing stopwords
Remove the stopwords.

In [36]:
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
for text in cleaned_X_train:
    words = [word for word in text.split() if word not in stop_words]
    cleaned_text = " ".join(words)
    cleaned_X_train = [" ".join([word for word in text.split() if word not in stop_words]) for text in cleaned_X_train]

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [39]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
for text in cleaned_X_train:
    words = [lemmatizer.lemmatize(word) for word in text.split()]
    cleaned_text = " ".join(words)
    cleaned_X_train = [" ".join([lemmatizer.lemmatize(word) for word in text.split()]) for text in cleaned_X_train]

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [44]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(cleaned_X_train)


## Extra features

In [47]:
data_train = pd.DataFrame({
    "preprocessed_text": cleaned_X_train
})

data_val = pd.DataFrame({
    "preprocessed_text": X_test.reset_index(drop=True)
})
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x))

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x))

data_train.head()

,preprocessed_text,money_mark,suspicious_words,text_len
0,ready pls call ops connect blair,0,0,32
1,fromamrekabir belloe emaila kabirsheecomtelaea...,0,1,1503
2,ok let mcguinness gate ffasi probably office e...,0,0,67
3,doc strategic dialogue press clip march doc st...,0,0,311
4,froma mre hoffman bancoleyaccrac ghanaeattn ai...,1,1,944


## How would work the Bag of Words with Count Vectorizer concept?

In [54]:
from sklearn.feature_extraction.text import CountVectorizer
X_bow = vectorizer.fit_transform(cleaned_X_train)
print(X_bow.shape)

(800, 16180)


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [56]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(cleaned_X_train)
print(X_tfidf.shape)

(800, 16180)


## And the Train a Classifier?

In [63]:
from sklearn.naive_bayes import MultinomialNB
classifier = MultinomialNB()
classifier.fit(X_tfidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[447.,353.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.58,-0.82]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 16180)","[[0. ,0.04,0. ,...,0. ,0.03,0. ], [0.09,0. ,1.05,...,0.08,0.55,0.09]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 16180)","[[-9.78,-9.74,-9.78,...,-9.78,-9.75,-9.78], [-9.8 ,-9.89,-9.17,...,-9.81,-9.45,-9.8 ]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,16180


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code